In [1]:
#!/usr/bin/env python3

# Phase 3 BONUS ABLATION: Evaluate Hindi Model without Positional Embeddings (No-RoPE)
#
**Self-contained** Kaggle evaluation notebook / script that runs the full Phase 2 evaluation suite
on the ablated Hindi Model H (trained from scratch with Rotary Position Embeddings completely removed):
#
1. **Intrinsic LM Metrics:** Test Loss, Perplexity (PPL), Bits-per-Byte (BPB)
2. **Tokenizer & Vocabulary Diagnostics:** Round-trip fidelity, bare combining diacritics audit
3. **Generation Quality:** BLEU-4, chrF++, ROUGE-L across temperatures (T = 0.0, 0.5, 1.0, 1.5)
4. **Script Validity & Constrained Decoding Studies:** Floating diacritic detection, repair ablation
5. **Diversity Diagnostics:** Repetition rates (1-4 gram), Distinct-1, Distinct-2
6. **Attention Analysis:** Heatmaps (L0, L4, L7), Per-head Entropy, Mean Distance summaries

In [2]:
import os

# === UPDATE THESE PATHS FOR KAGGLE OR LOCAL EXECUTION ===
CHECKPOINT_PATH = "/kaggle/input/notebooks/srinivasu2005/hindi-no-rope/checkpoints_hindi_no_rope/best_model.pt"

TEST_BIN = "/kaggle/input/notebooks/srinivasu2005/tokenize-hindi/hindi_tokenized/test.bin"

TEST_TEXT = "/kaggle/input/notebooks/srinivasu2005/hindi-tokenizer/splits/test.txt"

TOKENIZER_PATH = "/kaggle/input/notebooks/srinivasu2005/hindi-tokenizer/tokenizer/tokenizer.model"

OUTPUT_DIR = "/kaggle/working/eval_hindi_no_rope"

LANG = "Hindi (No-RoPE Ablation)"
VOCAB_SIZE = 8000
D_MODEL = 512
N_LAYERS = 8
N_HEADS = 8
D_FF = 1024       # Trained No-RoPE model configuration
MAX_SEQ_LEN = 512
DROPOUT = 0.0     # Eval mode
TIE_WEIGHTS = True
PAD_ID = 0; UNK_ID = 1; BOS_ID = 2; EOS_ID = 3

NUM_PREFIXES = 100
PREFIX_TOKENS = 32
MAX_NEW_TOKENS = 128
TEMPERATURES = [0.0, 0.5, 1.0, 1.5]
TOP_K = 50
BATCH_SIZE = 16
EVAL_MAX_BATCHES = None  # Full test set

In [3]:
import subprocess, sys

try:
    import sentencepiece
    import sacrebleu
    import rouge_score
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentencepiece", "sacrebleu", "rouge-score", "matplotlib"])

try:
    if not os.path.exists("/tmp/NotoSansDevanagari-Regular.ttf"):
        subprocess.check_call([
            "wget", "-q",
            "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansDevanagari/NotoSansDevanagari-Regular.ttf",
            "-O", "/tmp/NotoSansDevanagari-Regular.ttf"
        ])
except Exception as e:
    print(f"Warning: Could not fetch font. Error: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 2.3 MB/s eta 0:00:00


In [4]:
import math, json, time, logging, re
from dataclasses import dataclass
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

if os.path.exists('/tmp/NotoSansDevanagari-Regular.ttf'):
    fm.fontManager.addfont('/tmp/NotoSansDevanagari-Regular.ttf')
plt.rcParams['font.family'] = ['Noto Sans Devanagari', 'DejaVu Sans']

import sentencepiece as spm

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s — %(message)s")
logger = logging.getLogger("eval_hindi_no_rope")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
@dataclass
class TransformerConfig:
    vocab_size: int = VOCAB_SIZE
    d_model: int = D_MODEL
    n_layers: int = N_LAYERS
    n_heads: int = N_HEADS
    d_ff: int = D_FF
    max_seq_len: int = MAX_SEQ_LEN
    dropout: float = DROPOUT
    tie_weights: bool = TIE_WEIGHTS
    pad_id: int = PAD_ID
    unk_id: int = UNK_ID
    bos_id: int = BOS_ID
    eos_id: int = EOS_ID
    @property
    def d_k(self):
        return self.d_model // self.n_heads

class MultiHeadCausalSelfAttention(nn.Module):
    """
    Standard Multi-Head Causal Self-Attention with Positional Embeddings completely removed.
    Q and K dot-product relies strictly on causal masking without any rotary or additive positional offset.
    """
    def __init__(self, cfg):
        super().__init__()
        self.n_heads, self.d_k, self.d_model = cfg.n_heads, cfg.d_k, cfg.d_model
        self.W_Q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_K = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_V = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.W_O = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        # ONLY causal mask — NO RoPE cos/sin buffers
        self.register_buffer("causal_mask", torch.triu(torch.ones(cfg.max_seq_len, cfg.max_seq_len), diagonal=1).bool())

    def forward(self, x, return_attention=False):
        B, T, _ = x.shape
        Q = self.W_Q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        # Standard scaled dot-product WITHOUT positional rotation
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        scores = scores.masked_fill(self.causal_mask[:T, :T].unsqueeze(0).unsqueeze(0), float('-inf'))
        aw = torch.softmax(scores, dim=-1)
        ctx = torch.matmul(self.attn_dropout(aw), V).transpose(1, 2).contiguous().view(B, T, self.d_model)
        out = self.resid_dropout(self.W_O(ctx))
        return (out, aw) if return_attention else (out, None)

class SwiGLUFeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.W_gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.W_up = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.W_down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.dropout(self.W_down(F.silu(self.W_gate(x)) * self.W_up(x)))

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = MultiHeadCausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = SwiGLUFeedForward(cfg)

    def forward(self, x, return_attention=False):
        ao, aw = self.attn(self.ln1(x), return_attention)
        x = x + ao
        x = x + self.ffn(self.ln2(x))
        return x, aw

class DecoderOnlyTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.config = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.emb_dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_ln = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_weights:
            self.lm_head.weight = self.embedding.weight

    def forward(self, input_ids, targets=None, return_attention=False):
        x = self.emb_dropout(self.embedding(input_ids))
        all_attn = [] if return_attention else None
        for block in self.blocks:
            x, aw = block(x, return_attention)
            if return_attention and aw is not None:
                all_attn.append(aw)
        logits = self.lm_head(self.final_ln(x))
        result = {"logits": logits}
        if targets is not None:
            result["loss"] = F.cross_entropy(logits.view(-1, self.config.vocab_size), targets.view(-1), ignore_index=self.config.pad_id)
        if return_attention:
            result["attention_weights"] = all_attn
        return result

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=128, temperature=1.0, top_k=0, repetition_penalty=1.0,
                 mask_special_tokens=True,
                 suppress_tokens=None, prevent_empty_parens=False, open_paren_ids=None, close_paren_ids=None,
                 prevent_floating_diacritics=False, combining_start_ids=None, valid_host_ids=None):
        self.eval()
        gen = input_ids
        special_ids = [self.config.pad_id, getattr(self.config, 'unk_id', 1), self.config.bos_id]
        mask_special = [tid for tid in special_ids if tid is not None and 0 <= tid < self.config.vocab_size]

        for _ in range(max_new_tokens):
            ctx = gen if gen.shape[1] <= self.config.max_seq_len else gen[:, -self.config.max_seq_len:]
            logits = self.forward(ctx)["logits"][:, -1, :]

            # 1. Hard-mask special tokens (<pad>, <unk>, <bos>)
            if mask_special_tokens and mask_special:
                logits[:, mask_special] = float('-inf')

            # 2. Experimental: Hard-mask suppressed tokens (Latin subwords, empty parens)
            if suppress_tokens:
                logits[:, suppress_tokens] = float('-inf')

            # 3. Prevent empty parentheses right after opening
            if prevent_empty_parens and open_paren_ids and close_paren_ids:
                for b in range(input_ids.size(0)):
                    if gen[b, -1].item() in open_paren_ids:
                        logits[b, close_paren_ids] = float('-inf')

            # 4. Constrained Decoding: Prevent floating diacritics without valid host
            if prevent_floating_diacritics and combining_start_ids:
                for b in range(input_ids.size(0)):
                    last_tok = gen[b, -1].item()
                    if valid_host_ids is not None and last_tok not in valid_host_ids:
                        logits[b, combining_start_ids] = float('-inf')

            if repetition_penalty != 1.0:
                for b in range(input_ids.size(0)):
                    for token_id in set(gen[b].tolist()):
                        if logits[b, token_id] < 0:
                            logits[b, token_id] *= repetition_penalty
                        else:
                            logits[b, token_id] /= repetition_penalty

            if temperature == 0.0:
                nt = logits.argmax(dim=-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k > 0:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = float('-inf')
                nt = torch.multinomial(torch.softmax(logits, dim=-1), 1)
            gen = torch.cat([gen, nt], dim=1)
            if (nt == self.config.eos_id).all():
                break
        return gen

class TokenizedDataset(Dataset):
    def __init__(self, bin_path, seq_len=512):
        self.seq_len = seq_len
        self.data = np.memmap(bin_path, dtype=np.uint16, mode='r')
        self.n_samples = max(0, (len(self.data) - 1) // seq_len)
    def __len__(self):
        return self.n_samples
    def __getitem__(self, idx):
        start = idx * self.seq_len
        c = self.data[start:start+self.seq_len+1].astype(np.int64)
        return torch.from_numpy(c[:-1]), torch.from_numpy(c[1:])
    def get_total_tokens(self):
        return len(self.data)

In [6]:
logger.info(f"Loading checkpoint from {CHECKPOINT_PATH}...")
try:
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
except TypeError:
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)

# Merge config from checkpoint
cfg_dict = dict(ckpt.get("config", {})) if isinstance(ckpt, dict) and "config" in ckpt else {}
cfg_dict.setdefault("vocab_size", VOCAB_SIZE)
cfg_dict.setdefault("d_model", D_MODEL)
cfg_dict.setdefault("n_layers", N_LAYERS)
cfg_dict.setdefault("n_heads", N_HEADS)
cfg_dict.setdefault("d_ff", D_FF)
cfg_dict.setdefault("max_seq_len", MAX_SEQ_LEN)
cfg_dict.setdefault("dropout", DROPOUT)
cfg_dict.setdefault("tie_weights", TIE_WEIGHTS)
cfg_dict.setdefault("unk_id", UNK_ID)
cfg_dict.pop("rope_base", None)  # Ensure no RoPE parameter leaks in

config = TransformerConfig(**cfg_dict)
model = DecoderOnlyTransformer(config).to(device)

state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
model.load_state_dict(state_dict)
model.eval()
step_val = ckpt.get("step", "?") if isinstance(ckpt, dict) else "?"
val_loss_val = ckpt.get("val_loss", "?") if isinstance(ckpt, dict) else "?"
logger.info(f"Loaded No-RoPE model from step {step_val}, val_loss={val_loss_val}")

sp = spm.SentencePieceProcessor()
sp.load(TOKENIZER_PATH)
logger.info(f"Loaded tokenizer from {TOKENIZER_PATH} (vocab_size={sp.get_piece_size()})")

all_results = {"language": LANG, "checkpoint_step": ckpt.get("step", -1) if isinstance(ckpt, dict) else -1}

# ============================================================

[2026-09-16 13:54:40,454] INFO — Loading checkpoint from /kaggle/input/notebooks/srinivasu2005/hindi-no-rope/checkpoints_hindi_no_rope/best_model.pt...
[2026-09-16 13:54:43,661] INFO — Loaded No-RoPE model from step 16000, val_loss=3.4250033283233643
[2026-09-16 13:54:43,701] INFO — Loaded tokenizer from /kaggle/input/notebooks/srinivasu2005/hindi-tokenizer/tokenizer/tokenizer.model (vocab_size=8000)


In [7]:
# ============================================================
DEV_COMBINING_MARKS = set(
    [chr(cp) for cp in range(0x0901, 0x0904)] +  # chandrabindu (ँ), anusvara (ं), visarga (ः)
    [chr(0x093C)] +                              # nukta (़)
    [chr(cp) for cp in range(0x093E, 0x094E)] +  # matras (ा to ौ) + virama/halant (्)
    [chr(cp) for cp in range(0x0955, 0x0958)] +  # additional signs
    [chr(cp) for cp in range(0x0962, 0x0964)]    # vocalic signs
)
DEV_CONSONANTS = set([chr(cp) for cp in range(0x0915, 0x093A)] + [chr(cp) for cp in range(0x0958, 0x0960)])
DEV_INDEP_VOWELS = set([chr(cp) for cp in range(0x0904, 0x0915)])
DEV_HOST_CHARS = DEV_CONSONANTS | DEV_INDEP_VOWELS

# Identify tokens starting with a bare combining mark (orphaned diacritics)
combining_start_ids = [
    i for i in range(sp.get_piece_size())
    if sp.id_to_piece(i).lstrip(' ▁_ ') and sp.id_to_piece(i).lstrip(' ▁_ ')[0] in DEV_COMBINING_MARKS
]
# Identify tokens ending in a valid host character (consonant or independent vowel)
valid_host_ids = set([
    i for i in range(sp.get_piece_size())
    if sp.id_to_piece(i) and sp.id_to_piece(i)[-1] in DEV_HOST_CHARS
])
logger.info(f"[Diacritic Guard] {len(combining_start_ids)}/{sp.get_piece_size()} pieces start with bare combining marks; {len(valid_host_ids)} host tokens")

def repair_devanagari(text):
    """Backstop regex cleaner: removes orphaned combining marks that have no valid host."""
    combining_class = r'[\u0901-\u0903\u093C\u093E-\u094D\u094E-\u094F\u0955-\u0957\u0962-\u0963]'
    text = re.sub(r'(?:^|\n)' + combining_class + r'+', '', text)
    text = re.sub(r'([\s\(\)\[\]\{\}।,\.\?!:;\-"\'–—«»])' + combining_class + r'+', r'\1', text)
    matra_class = r'[\u093E-\u094C\u094E-\u094F\u0955-\u0957\u0962-\u0963]'
    text = re.sub(r'(' + matra_class + r')' + matra_class + r'+', r'\1', text)
    return re.sub(r' {2,}', ' ', text).strip()

def check_script_validity(text):
    """Detect invalid Devanagari sequences (floating matras, halant at word start, etc.)."""
    combining_class = r'[\u0901-\u0903\u093C\u093E-\u094D\u094E-\u094F\u0955-\u0957\u0962-\u0963]'
    matra_class = r'[\u093E-\u094C\u094E-\u094F\u0955-\u0957\u0962-\u0963]'
    errors = []
    if re.search(r'(?:^|\n)' + combining_class, text):
        errors.append("combining_at_start")
    if re.search(r'[\s\(\)\[\]।,\.\?!:;\-"\'–—]' + combining_class, text):
        errors.append("floating_after_space_or_punct")
    if re.search(r'(' + matra_class + r'){2,}', text):
        errors.append("consecutive_matras")
    return {"is_valid": len(errors) == 0, "error_count": len(errors), "error_types": errors}

# ============================================================

[2026-09-16 13:54:43,742] INFO — [Diacritic Guard] 273/8000 pieces start with bare combining marks; 4664 host tokens


In [8]:
# ============================================================
logger.info("\n" + "="*60 + "\n1. Intrinsic LM Metrics (No-RoPE Ablation)\n" + "="*60)

test_ds = TokenizedDataset(TEST_BIN, MAX_SEQ_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True, drop_last=False)

total_loss, total_tokens, n_batches = 0.0, 0, 0
with torch.no_grad():
    for x, y in test_loader:
        if EVAL_MAX_BATCHES and n_batches >= EVAL_MAX_BATCHES:
            break
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast('cuda', dtype=torch.float16):
            result = model(x, targets=y)
        non_pad = (y != PAD_ID).sum().item()
        total_loss += result["loss"].item() * non_pad
        total_tokens += non_pad
        n_batches += 1

avg_loss = total_loss / max(1, total_tokens)
ppl = math.exp(avg_loss) if avg_loss < 20 else float('inf')

num_bytes = 0
if os.path.exists(TEST_TEXT):
    with open(TEST_TEXT, "rb") as f:
        num_bytes = len(f.read())
bpb = (avg_loss * total_tokens / num_bytes * math.log2(math.e)) if num_bytes > 0 else None

lm_results = {
    "test_loss": round(avg_loss, 4),
    "test_ppl": round(ppl, 2),
    "test_bpb": round(bpb, 4) if bpb else None,
    "num_tokens_evaluated": total_tokens,
    "num_bytes": num_bytes
}
all_results["lm_metrics"] = lm_results
logger.info(f"  Test Loss: {avg_loss:.4f} | Test PPL: {ppl:.2f} | Test BPB: {bpb:.4f}" if bpb else f"  Test Loss: {avg_loss:.4f} | Test PPL: {ppl:.2f}")

# ============================================================

[2026-09-16 13:54:43,759] INFO — 
1. Intrinsic LM Metrics (No-RoPE Ablation)
[2026-09-16 13:57:20,641] INFO —   Test Loss: 3.4161 | Test PPL: 30.45 | Test BPB: 0.5289


In [9]:
# ============================================================
logger.info("\n" + "="*60 + "\n1b. Tokenizer & Vocabulary Diagnostics\n" + "="*60)

round_trip_tested = 0
round_trip_matches = 0
round_trip_failures = []
if os.path.exists(TEST_TEXT):
    with open(TEST_TEXT, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            s = line.strip()
            if not s: continue
            round_trip_tested += 1
            reconstructed = sp.decode(sp.encode(s, out_type=int))
            if reconstructed == s:
                round_trip_matches += 1
            elif len(round_trip_failures) < 5:
                round_trip_failures.append({"orig": s[:100], "reconstructed": reconstructed[:100]})
            if round_trip_tested >= 200: break
rt_fidelity = round(round_trip_matches / max(1, round_trip_tested) * 100, 2)
logger.info(f"  Round-trip test: {round_trip_matches}/{round_trip_tested} exact matches ({rt_fidelity}%)")

bare_combining_tokens = [i for i in range(sp.get_piece_size()) if sp.id_to_piece(i) and all(c in DEV_COMBINING_MARKS for c in sp.id_to_piece(i))]
vocab_diag = {
    "total_vocab_size": sp.get_piece_size(),
    "bare_combining_tokens_count": len(bare_combining_tokens),
    "bare_combining_tokens_pct": round(len(bare_combining_tokens) / sp.get_piece_size() * 100, 3),
    "starts_with_combining_count": len(combining_start_ids),
    "starts_with_combining_pct": round(len(combining_start_ids) / sp.get_piece_size() * 100, 3),
    "sample_bare_combining_tokens": [sp.id_to_piece(i) for i in bare_combining_tokens[:10]],
    "sample_starts_with_combining": [sp.id_to_piece(i) for i in combining_start_ids[:10]]
}
all_results["tokenizer_diagnostics"] = {
    "round_trip_fidelity_pct": rt_fidelity,
    "round_trip_samples_tested": round_trip_tested,
    "round_trip_failures": round_trip_failures,
    "vocab_composition": vocab_diag
}

# ============================================================

[2026-09-16 13:57:20,662] INFO — 
1b. Tokenizer & Vocabulary Diagnostics
[2026-09-16 13:57:20,685] INFO —   Round-trip test: 191/200 exact matches (95.5%)


In [10]:
# ============================================================
logger.info("\n" + "="*60 + "\n2. Generation Quality (No-RoPE Ablation)\n" + "="*60)

prefixes = []
if os.path.exists(TEST_TEXT):
    with open(TEST_TEXT, "r", encoding="utf-8") as f:
        doc = []
        for line in f:
            s = line.strip()
            if s: doc.append(s)
            elif doc:
                text = " ".join(doc)
                ids = sp.encode(text, out_type=int)
                if len(ids) >= PREFIX_TOKENS + MAX_NEW_TOKENS:
                    prefixes.append({
                        "prefix_ids": ids[:PREFIX_TOKENS],
                        "ref_text": sp.decode(ids[PREFIX_TOKENS:PREFIX_TOKENS+MAX_NEW_TOKENS])
                    })
                    if len(prefixes) >= NUM_PREFIXES: break
                doc = []
    logger.info(f"  Extracted {len(prefixes)} prefixes from test set")

gen_results = {}
for temp in TEMPERATURES:
    gen_ids_list = []
    gen_texts = []
    for p in prefixes:
        inp = torch.tensor([p["prefix_ids"]], dtype=torch.long, device=device)
        with torch.no_grad():
            out = model.generate(
                inp,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=temp,
                top_k=TOP_K if temp > 0 else 0,
                repetition_penalty=1.2,
                mask_special_tokens=True
            )
        gids = out[0, len(p["prefix_ids"]):].tolist()
        gen_ids_list.append(gids)
        gen_texts.append(sp.decode(gids))

    ref_texts = [p["ref_text"] for p in prefixes]
    metrics = {"num_samples": len(gen_texts)}
    try:
        import sacrebleu
        from rouge_score import rouge_scorer
        bleu = sacrebleu.corpus_bleu(gen_texts, [ref_texts])
        chrf = sacrebleu.corpus_chrf(gen_texts, [ref_texts], word_order=2)
        scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
        rl = [scorer.score(r, g)['rougeL'].fmeasure for r, g in zip(ref_texts, gen_texts)]
        metrics.update({
            "bleu4": round(bleu.score, 2),
            "chrf_pp": round(chrf.score, 2),
            "rouge_l": round(sum(rl) / len(rl) * 100, 2) if rl else 0
        })
    except Exception as e:
        logger.warning(f"  Metrics error: {e}")

    validity_checks = [check_script_validity(gt) for gt in gen_texts]
    valid_samples = sum(1 for v in validity_checks if v["is_valid"])
    total_errs = sum(v["error_count"] for v in validity_checks)
    metrics["script_validity_rate_pct"] = round(valid_samples / max(1, len(gen_texts)) * 100, 2)
    metrics["avg_script_errors_per_sample"] = round(total_errs / max(1, len(gen_texts)), 2)
    metrics["sample_generations"] = gen_texts[:5]
    metrics["sample_references"] = ref_texts[:5]
    metrics["generated_ids"] = gen_ids_list
    gen_results[f"temp_{temp}"] = metrics
    logger.info(f"  Temp={temp}: BLEU={metrics.get('bleu4','?')}, chrF++={metrics.get('chrf_pp','?')}, ROUGE-L={metrics.get('rouge_l','?')}, ScriptValid={metrics['script_validity_rate_pct']}%")

all_results["generation"] = gen_results

# ============================================================

[2026-09-16 13:57:20,720] INFO — 
2. Generation Quality (No-RoPE Ablation)
[2026-09-16 13:57:20,812] INFO —   Extracted 100 prefixes from test set
[2026-09-16 13:57:41,368] INFO — NumExpr defaulting to 4 threads.
[2026-09-16 13:57:42,287] INFO — Using default tokenizer.
[2026-09-16 13:57:42,292] INFO —   Temp=0.0: BLEU=0.02, chrF++=4.49, ROUGE-L=1.64, ScriptValid=93.0%
[2026-09-16 13:58:08,797] INFO — Using default tokenizer.
[2026-09-16 13:58:08,801] INFO —   Temp=0.5: BLEU=0.04, chrF++=6.19, ROUGE-L=2.1, ScriptValid=91.0%
[2026-09-16 13:59:05,402] INFO — Using default tokenizer.
[2026-09-16 13:59:05,407] INFO —   Temp=1.0: BLEU=0.49, chrF++=12.02, ROUGE-L=2.81, ScriptValid=90.0%
[2026-09-16 14:00:34,582] INFO — Using default tokenizer.
[2026-09-16 14:00:34,589] INFO —   Temp=1.5: BLEU=0.24, chrF++=15.48, ROUGE-L=0.9, ScriptValid=90.0%


In [11]:
# ============================================================
logger.info("\n" + "="*60 + "\n2b. Experimental Artifact Suppression Study\n" + "="*60)

latin_token_ids = [i for i in range(sp.get_piece_size()) if any(('a' <= c <= 'z' or 'A' <= c <= 'Z') for c in sp.id_to_piece(i))]
empty_paren_token_ids = [i for i in range(sp.get_piece_size()) if sp.id_to_piece(i).strip(' ▁') in ('()', '( )', '(-)', '( - )')]
experimental_suppress_tokens = list(set(latin_token_ids + empty_paren_token_ids))

open_paren_ids = set([i for i in range(sp.get_piece_size()) if sp.id_to_piece(i).strip(' ▁') == '('])
close_paren_ids = list(set([i for i in range(sp.get_piece_size()) if sp.id_to_piece(i).strip(' ▁') in (')', '),', ')।', ').')]))

def clean_generated_text(text):
    text = re.sub(r'\(\s*[-–—]?\s*\)', '', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

exp_results = {}
exp_temp = 1.0
exp_raw_texts, exp_suppressed_texts = [], []
for p in prefixes[:20]:
    inp = torch.tensor([p["prefix_ids"]], dtype=torch.long, device=device)
    with torch.no_grad():
        out_std = model.generate(inp, max_new_tokens=MAX_NEW_TOKENS, temperature=exp_temp, top_k=TOP_K, repetition_penalty=1.2, mask_special_tokens=True)
    exp_raw_texts.append(sp.decode(out_std[0, len(p["prefix_ids"]):].tolist()))

    with torch.no_grad():
        out_exp = model.generate(
            inp,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=exp_temp,
            top_k=TOP_K,
            repetition_penalty=1.2,
            mask_special_tokens=True,
            suppress_tokens=experimental_suppress_tokens,
            prevent_empty_parens=True,
            open_paren_ids=open_paren_ids,
            close_paren_ids=close_paren_ids
        )
    gids_exp = out_exp[0, len(p["prefix_ids"]):].tolist()
    exp_suppressed_texts.append(clean_generated_text(sp.decode(gids_exp)))

exp_results[f"comparison_temp_{exp_temp}"] = {
    "num_samples": len(exp_raw_texts),
    "standard_generation_samples": exp_raw_texts[:3],
    "suppressed_generation_samples": exp_suppressed_texts[:3],
    "num_latin_tokens_suppressed": len(latin_token_ids),
    "num_empty_paren_tokens_suppressed": len(empty_paren_token_ids)
}
all_results["experimental_artifact_suppression"] = exp_results

# ============================================================

[2026-09-16 14:00:34,612] INFO — 
2b. Experimental Artifact Suppression Study


In [12]:
# ============================================================
logger.info("\n" + "="*60 + "\n2c. Constrained Decoding Study (Diacritic Mitigation)\n" + "="*60)

constrained_study_results = {}
study_temps = [0.0, 0.5, 1.0]
num_study_samples = min(20, len(prefixes))

for stemp in study_temps:
    comp_samples = []
    total_raw_errs = 0
    total_mitigated_errs = 0
    raw_valid_count = 0
    mitigated_valid_count = 0

    for p in prefixes[:num_study_samples]:
        inp = torch.tensor([p["prefix_ids"]], dtype=torch.long, device=device)

        with torch.no_grad():
            out_raw = model.generate(
                inp, max_new_tokens=MAX_NEW_TOKENS, temperature=stemp,
                top_k=TOP_K if stemp > 0 else 0, repetition_penalty=1.2,
                mask_special_tokens=True
            )
        raw_text = sp.decode(out_raw[0, len(p["prefix_ids"]):].tolist())
        raw_val = check_script_validity(raw_text)
        total_raw_errs += raw_val["error_count"]
        if raw_val["is_valid"]: raw_valid_count += 1

        with torch.no_grad():
            out_constrained = model.generate(
                inp, max_new_tokens=MAX_NEW_TOKENS, temperature=stemp,
                top_k=TOP_K if stemp > 0 else 0, repetition_penalty=1.2,
                mask_special_tokens=True,
                prevent_floating_diacritics=True,
                combining_start_ids=combining_start_ids,
                valid_host_ids=valid_host_ids
            )
        constrained_text = sp.decode(out_constrained[0, len(p["prefix_ids"]):].tolist())
        repaired_text = repair_devanagari(constrained_text)
        mit_val = check_script_validity(repaired_text)
        total_mitigated_errs += mit_val["error_count"]
        if mit_val["is_valid"]: mitigated_valid_count += 1

        if len(comp_samples) < 5:
            comp_samples.append({
                "prefix": sp.decode(p["prefix_ids"]),
                "raw_generation": raw_text,
                "raw_is_valid": raw_val["is_valid"],
                "mitigated_generation": repaired_text,
                "mitigated_is_valid": mit_val["is_valid"],
                "reference": p["ref_text"][:100]
            })

    constrained_study_results[f"temp_{stemp}"] = {
        "samples_evaluated": num_study_samples,
        "raw_script_validity_pct": round(raw_valid_count / max(1, num_study_samples) * 100, 2),
        "raw_total_errors": total_raw_errs,
        "mitigated_script_validity_pct": round(mitigated_valid_count / max(1, num_study_samples) * 100, 2),
        "mitigated_total_errors": total_mitigated_errs,
        "comparison_examples": comp_samples
    }
    logger.info(f"  Temp={stemp}: Raw Validity={constrained_study_results[f'temp_{stemp}']['raw_script_validity_pct']}% -> Mitigated Validity={constrained_study_results[f'temp_{stemp}']['mitigated_script_validity_pct']}%")

all_results["constrained_decoding_study"] = constrained_study_results

# ============================================================

[2026-09-16 14:01:08,652] INFO — 
2c. Constrained Decoding Study (Diacritic Mitigation)
[2026-09-16 14:01:16,903] INFO —   Temp=0.0: Raw Validity=100.0% -> Mitigated Validity=100.0%
[2026-09-16 14:01:29,066] INFO —   Temp=0.5: Raw Validity=100.0% -> Mitigated Validity=100.0%
[2026-09-16 14:01:46,696] INFO —   Temp=1.0: Raw Validity=100.0% -> Mitigated Validity=100.0%


In [13]:
# ============================================================
logger.info("\n" + "="*60 + "\n3. Diversity Diagnostics\n" + "="*60)

def get_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def repetition_rate(tokens, n=4):
    ng = get_ngrams(tokens, n)
    if not ng: return 0.0
    c = Counter(ng)
    return sum(v-1 for v in c.values() if v > 1) / len(ng)

def distinct_n(tokens, n):
    ng = get_ngrams(tokens, n)
    return len(set(ng)) / len(ng) if ng else 0.0

diversity_results = {}
for temp_key, gdata in gen_results.items():
    all_gen = gdata.get("generated_ids", [])
    rep = {f"rep_{n}gram": round(np.mean([repetition_rate(g, n) for g in all_gen if len(g) > n]), 4) for n in [1, 2, 3, 4]}
    dist = {f"distinct_{n}": round(np.mean([distinct_n(g, n) for g in all_gen if len(g) > n]), 4) for n in [1, 2]}
    diversity_results[temp_key] = {**rep, **dist}
    logger.info(f"  {temp_key}: D1={dist['distinct_1']:.3f}, D2={dist['distinct_2']:.3f}, Rep4={rep['rep_4gram']:.3f}")

all_results["diversity"] = diversity_results

# ============================================================

[2026-09-16 14:01:46,717] INFO — 
3. Diversity Diagnostics
[2026-09-16 14:01:46,723] INFO —   temp_0.0: D1=0.882, D2=0.917, Rep4=0.074
[2026-09-16 14:01:46,729] INFO —   temp_0.5: D1=0.887, D2=0.946, Rep4=0.029
[2026-09-16 14:01:46,739] INFO —   temp_1.0: D1=0.879, D2=0.988, Rep4=0.000
[2026-09-16 14:01:46,754] INFO —   temp_1.5: D1=0.860, D2=0.991, Rep4=0.000


In [14]:
# ============================================================
logger.info("\n" + "="*60 + "\n4. Attention Analysis (No-RoPE)\n" + "="*60)

attn_dir = os.path.join(OUTPUT_DIR, "attention")
os.makedirs(attn_dir, exist_ok=True)

if prefixes:
    sample_ids = prefixes[0]["prefix_ids"][:40]
    sample_tokens = [sp.id_to_piece(i) for i in sample_ids]
    inp = torch.tensor([sample_ids], dtype=torch.long, device=device)

    with torch.no_grad():
        result = model(inp, return_attention=True)
    attn_weights_all = result["attention_weights"]

    # Heatmaps for early, middle, late layers
    layers_to_plot = [0, N_LAYERS // 2, N_LAYERS - 1]
    for li in layers_to_plot:
        aw = attn_weights_all[li]  # (1, h, T, T)
        for hi in range(min(N_HEADS, 4)):
            a = aw[0, hi].cpu().numpy()
            fig, ax = plt.subplots(figsize=(10, 8))
            ax.imshow(a, cmap='Blues', aspect='auto', vmin=0)
            if len(sample_tokens) <= 40:
                ax.set_xticks(range(len(sample_tokens)))
                ax.set_yticks(range(len(sample_tokens)))
                ax.set_xticklabels(sample_tokens, rotation=90, fontsize=6)
                ax.set_yticklabels(sample_tokens, fontsize=6)
            ax.set_xlabel("Key")
            ax.set_ylabel("Query")
            ax.set_title(f"{LANG} Layer {li}, Head {hi}")
            plt.colorbar(ax.images[0], ax=ax)
            plt.tight_layout()
            plt.savefig(os.path.join(attn_dir, f"heatmap_L{li}_H{hi}.png"), dpi=150)
            plt.close()

    # Per-head Entropy
    entropy_data = {}
    for li, aw in enumerate(attn_weights_all):
        a = aw.clamp(min=1e-10)
        ent = -(a * a.log()).sum(dim=-1)
        for hi in range(N_HEADS):
            entropy_data[f"L{li}_H{hi}"] = round(ent[0, hi].mean().item(), 4)

    # Per-head Mean Attention Distance
    distance_data = {}
    for li, aw in enumerate(attn_weights_all):
        T = aw.shape[-1]
        pos = torch.arange(T, device=device, dtype=torch.float32)
        dist_mat = (pos.unsqueeze(0) - pos.unsqueeze(1)).abs()
        md = (aw * dist_mat.unsqueeze(0).unsqueeze(0)).sum(dim=-1)
        for hi in range(N_HEADS):
            distance_data[f"L{li}_H{hi}"] = round(md[0, hi].mean().item(), 4)

    # Summary heatmaps
    for data, name, cmap, label in [
        (entropy_data, "entropy_summary", "YlOrRd", "Entropy (nats)"),
        (distance_data, "distance_summary", "viridis", "Mean Distance (tokens)")
    ]:
        mat = np.zeros((N_LAYERS, N_HEADS))
        for k, v in data.items():
            l, h = int(k.split("_")[0][1:]), int(k.split("_")[1][1:])
            mat[l, h] = v
        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(mat, cmap=cmap, aspect='auto')
        ax.set_xlabel("Head")
        ax.set_ylabel("Layer")
        ax.set_title(f"{LANG} {label}")
        ax.set_xticks(range(N_HEADS))
        ax.set_yticks(range(N_LAYERS))
        plt.colorbar(im, ax=ax, label=label)
        plt.tight_layout()
        plt.savefig(os.path.join(attn_dir, f"{name}.png"), dpi=150)
        plt.close()

    all_results["attention"] = {"entropy": entropy_data, "mean_distance": distance_data}
    logger.info(f"  Saved attention plots to {attn_dir}/")

# ============================================================

[2026-09-16 14:01:46,776] INFO — 
4. Attention Analysis (No-RoPE)
[2026-09-16 14:01:54,393] INFO —   Saved attention plots to /kaggle/working/eval_hindi_no_rope/attention/


In [15]:
# ============================================================
for tk in list(gen_results.keys()):
    if "generated_ids" in gen_results[tk]:
        del gen_results[tk]["generated_ids"]

results_path = os.path.join(OUTPUT_DIR, "eval_results.json")
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)
logger.info(f"\nAll results saved to {results_path}")

# ============================================================

[2026-09-16 14:01:54,413] INFO — 
All results saved to /kaggle/working/eval_hindi_no_rope/eval_results.json


In [16]:
# ============================================================
print("\n" + "="*60)
print(f"  {LANG} Model Evaluation Summary")
print("="*60)
print(f"  Test Loss:  {lm_results['test_loss']}")
print(f"  Test PPL:   {lm_results['test_ppl']}")
print(f"  Test BPB:   {lm_results.get('test_bpb', 'N/A')}")
print("-"*60)
if "tokenizer_diagnostics" in all_results:
    td = all_results["tokenizer_diagnostics"]
    vc = td["vocab_composition"]
    print(f"  Tokenizer Round-Trip Fidelity: {td['round_trip_fidelity_pct']}%")
    print(f"  Bare Combining Tokens in Vocab: {vc['bare_combining_tokens_count']} ({vc['bare_combining_tokens_pct']}%)")
    print(f"  Tokens Starting with Combining: {vc['starts_with_combining_count']} ({vc['starts_with_combining_pct']}%)")
print("-"*60)
print("  Raw Generation Metrics:")
for tk, m in gen_results.items():
    sv = m.get('script_validity_rate_pct', 'N/A')
    print(f"    {tk}: BLEU={m.get('bleu4','?')}, chrF++={m.get('chrf_pp','?')}, ROUGE-L={m.get('rouge_l','?')}, ScriptValid={sv}%")
print("-"*60)
if "constrained_decoding_study" in all_results:
    print("  Constrained Decoding Ablation (Diacritic Mitigation):")
    for tk, cs in all_results["constrained_decoding_study"].items():
        print(f"    {tk}: Raw Valid={cs['raw_script_validity_pct']}% -> Mitigated Valid={cs['mitigated_script_validity_pct']}% (Errs: {cs['raw_total_errors']} -> {cs['mitigated_total_errors']})")
print("-"*60)
for tk, d in diversity_results.items():
    print(f"  {tk}: D1={d['distinct_1']:.3f}, D2={d['distinct_2']:.3f}, Rep4={d['rep_4gram']:.4f}")
print("="*60)


  Hindi (No-RoPE Ablation) Model Evaluation Summary
  Test Loss:  3.4161
  Test PPL:   30.45
  Test BPB:   0.5289
------------------------------------------------------------
  Tokenizer Round-Trip Fidelity: 95.5%
  Bare Combining Tokens in Vocab: 24 (0.3%)
  Tokens Starting with Combining: 273 (3.413%)
------------------------------------------------------------
  Raw Generation Metrics:
    temp_0.0: BLEU=0.02, chrF++=4.49, ROUGE-L=1.64, ScriptValid=93.0%
    temp_0.5: BLEU=0.04, chrF++=6.19, ROUGE-L=2.1, ScriptValid=91.0%
    temp_1.0: BLEU=0.49, chrF++=12.02, ROUGE-L=2.81, ScriptValid=90.0%
    temp_1.5: BLEU=0.24, chrF++=15.48, ROUGE-L=0.9, ScriptValid=90.0%
------------------------------------------------------------
  Constrained Decoding Ablation (Diacritic Mitigation):
    temp_0.0: Raw Valid=100.0% -> Mitigated Valid=100.0% (Errs: 0 -> 0)
    temp_0.5: Raw Valid=100.0% -> Mitigated Valid=100.0% (Errs: 0 -> 0)
    temp_1.0: Raw Valid=100.0% -> Mitigated Valid=100.0% (Errs: 0 